# Retrieval-Augmented Generation (RAG) System for Financial Question Answering
### Consolidated Project Report

---

## 1. Abstract

Large Language Models (LLMs) often **hallucinate** when asked factual questions about a
company's financials. This project builds a **Retrieval-Augmented Generation (RAG)** pipeline
that grounds every answer in retrieved source passages, and then **measures** how faithful and
complete those answers are. The design follows the best practices from
**Wang et al. (2024), "Searching for Best Practices in Retrieval-Augmented Generation" (EMNLP 2024)**,
and is evaluated on the **RAGBench-FinQA** benchmark against ground-truth scores using the four
**TRACe** metrics plus an **LLM-as-a-Judge** check. The entire system runs in-process on a single
Kaggle **T4 GPU (~15 GB)** with no paid API keys.

**Headline result (N ≈ 1000):** average TRACe **0.668 vs 0.494** ground truth (**+0.174**),
context relevance **0.724** and context utilization **0.765**, with a **72.0% PASS rate**
(720 / 1000) from an LLM-as-a-Judge.

---

## 2. Problem Statement & Objectives

**Problem.** Financial Q&A demands answers that are (a) *grounded* in real documents and
(b) *verifiable*. A plain LLM cannot guarantee either.

**Objectives.**
1. Build a complete, modular RAG pipeline for financial Q&A.
2. Implement the EMNLP-2024 "best-practice" components (hybrid retrieval, HyDE, reranking,
   repacking, compression, query classification).
3. Evaluate answer quality **quantitatively** (TRACe metrics) and **qualitatively**
   (LLM-as-a-Judge), and compare against the RAGBench ground truth **at scale (≈ 1000 samples)**.
4. Make the whole system reproducible on free hardware (single T4 GPU, no API keys).

---

## 3. Dataset

**RAGBench-FinQA** (`galileo-ai/ragbench`, config `finqa`) — 12,502 financial Q&A samples.
This run used **`SAMPLE_SIZE = 1000`** (seed 42) sampled from the full set. Each sample contains:

| Field | Meaning | Use in this project |
|-------|---------|---------------------|
| `question` | User query | Input to the pipeline |
| `documents` / `context` | Pre-retrieved passages | Corpus to retrieve/generate over |
| `response` | Reference (GPT-3.5/Haiku) answer | Reference for correctness/completeness |
| `relevance_score`, `utilization_score`, `completeness_score`, `adherence_score` | Ground-truth TRACe scores | Benchmark to compare our scores against |

Exploratory Data Analysis (EDA) plots the ground-truth score distributions and the
question/context length statistics before any modelling.

> **Key framing:** the goal is **not** to reproduce the reference answers. We build our *own*
> pipeline, generate our *own* answers, compute our *own* TRACe scores, and measure **how close
> we get to the ground truth**.

---

## 4. System Architecture

```
Question
 → [1] Query classification .... does it even need retrieval? (skip trivial queries)
 → [2] Chunk .................... split documents into ~512-token pieces
 → [3] Embed ................... BAAI/llm-embedder (768-d)
 → [4] Index ................... FAISS vector store
 → [5] Retrieve + HyDE ......... BM25 + dense, fused with Reciprocal Rank Fusion (RRF)
 → [6] Rerank .................. monoT5 cross-encoder (precision)
 → [7] Repack .................. reverse order (most-relevant chunk last)
 → [8] Compress ................ Recomp-style extractive (drop noise)
 → [9] Generate ................ Qwen2.5-7B-Instruct (4-bit), grounded prompt
 → [10] Evaluate ............... 4 TRACe metrics (NLI entailment) + LLM-as-a-Judge
 → Answer + scores
```

**Final component choices and the reasoning behind each:**

| Component | Choice | Why this choice |
|-----------|--------|-----------------|
| Embedder | `BAAI/llm-embedder` (768-d) | EMNLP-2024 quality/size sweet spot (instruction-tuned) |
| Reranker | `castorini/monot5-base-msmarco-10k` | Paper-recommended cross-encoder reranker |
| Generator | `Qwen2.5-7B-Instruct` (4-bit nf4) | Strong instruction model that fits a 15 GB T4 |
| Chunk / overlap | 512 / 128 tokens | Safe for the context window, preserves continuity |
| Top-k / fetch-k | 4 / 15 | Fetch wide, keep sharp → higher precision |
| Fusion | RRF (k = 60) | Robust way to merge BM25 + dense rankings |
| Compression | Recomp, ≤ 10 sentences (min 0.2) | Removes irrelevant sentences before generation |
| Temperature | 0.1 | Factual, low-variance answers |
| Evaluator NLI | `cross-encoder/nli-deberta-v3-base` | Sentence-level entailment for TRACe |

---

## 5. Methods Tried — and Why

This is the core of the project: rather than accepting a single design, each stage was
implemented with **multiple interchangeable strategies** so they could be compared. Below is
what was tried at every stage and the reasoning.

### 5.1 Chunking strategy
- **Fixed-size** — simplest baseline; can cut sentences mid-thought.
- **Sliding window (chosen default, 512/128)** — overlap preserves context across boundaries so
  a fact split across two chunks is not lost.
- **Semantic chunking** — split on meaning; more faithful but slower and more complex.
- **Why:** financial passages contain tables and multi-sentence facts; the sliding window gave
  the best trade-off between context preservation and cost.

### 5.2 Retrieval strategy
- **BM25 only** — lexical/keyword match; strong for exact figures and tickers, blind to synonyms.
- **Dense only** — semantic embeddings; captures meaning, can miss exact numbers.
- **Hybrid BM25 + Dense with RRF (chosen)** — fuses both rankings; keyword precision *and*
  semantic recall.
- **Why:** finance questions mix exact numbers ("$2.74B") with semantic phrasing ("net revenue"),
  so neither retriever alone is sufficient — the hybrid captures both.

### 5.3 HyDE (Hypothetical Document Embeddings)
- The LLM first drafts a *hypothetical answer*, which is embedded and used to retrieve.
- **Why:** a hypothetical answer is often closer in embedding space to the true supporting
  passage than the short question is, improving recall on sparse queries.

### 5.4 Reranking (monoT5 cross-encoder)
- After fetching `fetch_k = 15` candidates, a cross-encoder re-scores each (query, passage) pair
  and keeps the top `top_k = 4`. Falls back to a MiniLM cross-encoder if monoT5 fails to load.
- **Why:** bi-encoder retrieval is fast but coarse; a cross-encoder reads the pair jointly and is
  far more precise, so the generator sees only the best evidence.

### 5.5 Repacking (reverse order)
- The most-relevant chunk is placed **last** in the prompt.
- **Why:** LLMs suffer from the "lost-in-the-middle" effect; the paper found that putting the
  strongest evidence nearest the question improves grounding.

### 5.6 Contextual compression (Recomp-style)
- Extractive filter keeps ≤ 10 of the most relevant sentences (min score 0.2) before generation.
- **Why:** removes distracting/irrelevant sentences, reduces prompt length and cost, and lifts
  adherence by not giving the model noise to latch onto.

### 5.7 Query classification
- A lightweight heuristic decides whether a query even needs retrieval (chit-chat / pure
  arithmetic → skip; everything else → retrieve).
- **Why:** trivial queries waste retrieval + generation budget; skipping them saves latency.

### 5.8 Generator model — the biggest iteration story
- **Attempt 1 — Ollama + Qwen2.5-14B:** originally planned to serve a 14B model via Ollama.
  Not portable to Kaggle (no Ollama server), so switched to in-process Hugging Face.
- **Attempt 2 — Qwen2.5-14B (4-bit) in-process:** stronger, but its fp16 **load-time peak
  overflows a 15 GB T4** → CUDA out-of-memory. Needs ≥ 24 GB.
- **Final — Qwen2.5-7B-Instruct (4-bit nf4):** the largest model that loads *reliably* on a
  single T4, with strong instruction-following. A grounded system prompt forbids outside
  knowledge and strips "Based on the context…" preambles that hurt NLI adherence scoring.
- **Why it matters:** documents a real hardware-vs-quality trade-off and the reasoning that led
  to the final, reproducible choice.

### 5.9 Evaluation methods
- **TRACe (4 metrics, NLI-based, chosen primary):** sentence-level entailment (see §6).
- **LLM-as-a-Judge (secondary, holistic):** an LLM reads question + context + our answer +
  reference and scores faithfulness/correctness/completeness (1–5) with a PASS/FAIL verdict,
  returned as strict JSON.
- **Why two:** TRACe is objective and reproducible but mechanical; the judge adds a holistic,
  human-like sanity check. Using both cross-validates the results.

### 5.10 Judge model selection (another T4-driven decision)
- **14B judge:** best quality but OOM on a T4 even after freeing the generator → rejected.
- **Independent 3B judge (`Qwen2.5-3B-Instruct`):** small enough to load *alongside* the 7B; no
  self-bias — exposed via `USE_INDEPENDENT_JUDGE`.
- **Self-judge (reuse the loaded 7B — used in this run):** zero extra VRAM, no extra download,
  most reliable on free hardware. The judge cell also self-heals (reloads the 7B if an earlier
  run freed its VRAM) and defensively frees partially-loaded weights before any fallback.
- **Why:** balances independence against the hard 15 GB memory ceiling.

---

## 6. Evaluation Methodology — the 4 TRACe Metrics

All metrics are computed with an NLI cross-encoder as **proportions of entailed sentences**
(entailment threshold 0.5), scored **per sentence pair** to avoid long-context truncation:

| Metric | Question it answers | How it is computed |
|--------|---------------------|--------------------|
| **Context Relevance** | Did retrieval fetch supporting passages? | Fraction of context sentences that entail the reference answer (falls back to the query) |
| **Context Utilization** | Did the LLM actually use the relevant context? | Of the *relevant* context, the fraction reflected in the answer |
| **Completeness** | Does the answer cover the ground-truth answer? | Fraction of GT-answer sentences entailed by our answer |
| **Adherence** | Is the answer grounded (no hallucination)? | Fraction of answer sentences entailed by some context sentence |

`avg_trace` is the mean of the four. Interpretation: **> 0.80 good · 0.60–0.80 acceptable ·
< 0.60 needs work.**

> **Notable correctness fix.** An early version computed *completeness* and *adherence*
> identically (both `context → response`). This was corrected so completeness measures
> `response → GT-answer` (coverage) while adherence measures `context → response` (grounding) —
> two genuinely different things.

---

## 7. Results

Results below are from the executed run on **`SAMPLE_SIZE = 1000`** FinQA samples (full
EMNLP-2024 best-practice pipeline: hybrid retrieval + HyDE + monoT5 rerank + reverse repack +
Recomp compression + Qwen2.5-7B generator). Comparison rows were saved to
`phase1_baseline_results.csv` (1066 rows after the per-question GT merge).

**7.1 Our pipeline vs RAGBench ground truth** (N ≈ 1000):

| Metric | Our score | RAGBench GT | Δ (ours − GT) |
|--------|-----------|-------------|---------------|
| Context Relevance | 0.724 | 0.078 | ▲ 0.646 |
| Context Utilization | 0.765 | 0.065 | ▲ 0.700 |
| Completeness | 0.556 | 0.910 | ▼ 0.355 |
| Adherence | 0.629 | 0.923 | ▼ 0.294 |
| **avg_trace** | **0.668** | **0.494** | **▲ 0.174** |

*Reading it:* our retrieval quality (context relevance/utilization) is far higher than the
sparse RAGBench GT columns, while completeness and adherence trail the reference — i.e. the
pipeline retrieves and uses strong evidence, but sometimes under-covers the reference answer or
slightly over-states beyond the retrieved context. Overall `avg_trace` **0.668 vs 0.494**
(**+0.174**) beats the ground-truth baseline across ~1000 samples.

**7.2 LLM-as-a-Judge** (N = 1000, self-judge = Qwen2.5-7B-Instruct):

| Dimension | Score |
|-----------|-------|
| PASS rate | **72.0%** (720 / 1000) |
| Faithfulness | 4.22 / 5 |
| Correctness | 4.02 / 5 |
| Completeness | 3.99 / 5 |

The judge broadly agrees with the TRACe metrics: answers are mostly faithful (4.22) and correct
(4.02), with completeness the weakest dimension (3.99) — consistent with the lower TRACe
completeness. (Two of the 1000 judge responses failed strict-JSON parsing and were scored as
FAIL, a minor conservative bias.)

**7.3 Latency / cost.** Per-sample cost is dominated by LLM generation plus the extra HyDE draft
per query. At the 1000-sample scale this is a multi-hour, compute-bound sweep on a single T4
(the LLM-as-a-Judge pass alone over 1000 answers took roughly 80 minutes).

> _Note: a controlled component ablation (BM25-only vs Dense-only vs Hybrid vs +rerank vs +HyDE)
> was designed and is toggleable in code, but was not run as a separate batch in this pass._

---

## 8. Engineering Challenges & Solutions

| Challenge | Root cause | Solution |
|-----------|-----------|----------|
| 14B model / independent judge CUDA OOM | fp16 load-time peak > 15 GB T4 | Use 7B (4-bit) generator; self-judge or 3B independent judge |
| VRAM stayed pinned after a failed load | Exception traceback holds partial weights | `del exc; gc.collect(); torch.cuda.empty_cache()` before retry |
| Judge freed the generator's VRAM | Optional `FREE_GENERATOR_BEFORE_JUDGE` releases weights | Judge cell self-heals — reloads the 7B if `generator.model is None` |
| Ollama not available on Kaggle | No local server on the platform | Switched to in-process Hugging Face `transformers` |
| NLI truncating long financial context | 3000-token premise > 512 model limit | Score each sentence pair individually, take per-sentence max |
| monoT5 fails to download | Network / model availability | Graceful fallback to a MiniLM cross-encoder reranker |
| FAISS-GPU package unpublished | `faiss-gpu` no longer on PyPI | Try GPU, gracefully fall back to `faiss-cpu` (pre-installed on Kaggle) |
| Judge output not valid JSON | LLM adds extra text after the JSON | Regex-extract the first JSON object; safe FAIL fallback on parse error |

---

## 9. Interactive Streamlit Playground

To make the pipeline usable by non-programmers and to demonstrate every strategy live, the
project ships an interactive **Streamlit** web app. It lets anyone type a question, paste (or
pick) context, toggle each best-practice component on/off, tune the knobs, and immediately see
the grounded answer together with its TRACe scores and an LLM-as-a-Judge verdict.

### 9.1 Purpose
- Turn the notebook pipeline into a point-and-click **playground** — no code required.
- Show the *effect* of each EMNLP-2024 component by switching it on/off and re-running.
- Provide a clean demo surface for the project submission / viva.

### 9.2 How it is built (single source of truth)
The notebook itself **generates** a fully self-contained `app.py` (Step 3.1). Rather than
importing from the notebook, the entire pipeline (chunker, embedder, FAISS store, BM25/Dense/
Hybrid retrievers, HyDE, monoT5 reranker, reverse-repack, Recomp compressor, generator, TRACe
evaluator, and the LLM-as-a-Judge) is embedded verbatim into `app.py` as a raw string. This
guarantees the UI always matches the notebook code and — importantly — works on Kaggle, where
`inspect.getsource()` fails because the classes live in `__main__` with no source file. The cell
also exports 50 FinQA rows to `finqa_examples.json` so the UI has ready-made dataset examples.

### 9.3 What the user controls
- **Strategy selection:** chunking (`sliding_window` / `semantic` / `fixed`), retrieval mode
  (`Hybrid (BM25+Dense)` / `Dense only` / `BM25 only`), and independent checkboxes for **HyDE**,
  **monoT5 rerank**, **reverse repack**, **Recomp compression**, and the **query classifier**.
- **Parameter knobs:** `top_k`, `fetch_k`, `chunk_size`, `overlap`, `temperature`,
  `max_new_tokens`.
- **One-click presets:** *Paper best-practice* (everything on), *Fast / cheap* (dense-only, no
  rerank/HyDE), and *BM25 baseline* — so users can compare configurations instantly.
- **Evaluation toggles:** run TRACe metrics and/or the LLM-as-a-Judge, plus an optional
  reference answer for completeness/correctness scoring.

### 9.4 What the user sees
- The **grounded answer**, with a per-run **latency** and the number of context chunks used.
- **TRACe scores** as five live metric tiles + a bar chart (context relevance, utilization,
  completeness, adherence, avg_trace).
- The **LLM-as-a-Judge** verdict (PASS/FAIL with Faithfulness / Correctness / Completeness).
- Expandable panels for the **retrieved context** and a **pipeline trace** listing every stage
  that actually ran (chunking → classifier → retrieval → repack → compression → generation → eval).

### 9.5 Deployment on Kaggle (T4-aware design)
The app runs as a **separate process** and loads its **own** copy of the models via
`@st.cache_resource`. Because a single 15 GB T4 cannot hold two copies, the notebook first
**frees its own VRAM** (Step 3.0) — or the user restarts the kernel — so the app owns the GPU.
Since Kaggle blocks `localhost`, the app is exposed through a **Cloudflare quick-tunnel** (no
token needed), with **ngrok** documented as an alternative. Robustness touches: the app falls
back to a MiniLM reranker if monoT5 fails to load, and to built-in starter questions if the
dataset examples are missing.

---

## 10. Conclusion & Future Work

The project delivers a complete, reproducible, best-practice RAG pipeline for financial Q&A that
runs on free hardware and is evaluated at scale (≈ 1000 samples) both quantitatively (TRACe vs
ground truth) and qualitatively (LLM-as-a-Judge). The design consistently favoured choices that
were justified either by the EMNLP-2024 findings or by the hard 15 GB memory ceiling. The
pipeline clearly out-retrieves the ground-truth baseline (relevance/utilization) while
completeness and adherence remain the main areas to improve.

**Future work.**
- Run the full 12,502-sample evaluation and a controlled per-component ablation.
- Improve completeness/adherence (e.g. answer-planning prompts, stricter grounding, better
  compression tuning).
- Try a larger generator/judge on ≥ 24 GB hardware and re-measure.
- Add an independent judge (3B) run to remove any self-judge bias.

---

## 11. References

1. Wang et al. (2024). *Searching for Best Practices in Retrieval-Augmented Generation.* EMNLP 2024.
2. Galileo-AI. *RAGBench* dataset (`finqa` configuration).
3. TRACe evaluation framework (Context Relevance, Utilization, Completeness, Adherence).
4. Gao et al. *Precise Zero-Shot Dense Retrieval without Relevance Labels* (HyDE).
5. Nogueira et al. *Document Ranking with a Pretrained Sequence-to-Sequence Model* (monoT5).
6. Xu et al. *RECOMP: Improving RAG with Compression and Selective Augmentation.*
7. Qwen Team. *Qwen2.5-Instruct* model family.


